In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
events_path = BASE_DIR / "data/clean/events.parquet"

In [3]:
events_data = pd.read_parquet(events_path)

In [4]:
events_data['event_datetime'] = pd.to_datetime(events_data.event_timestamp, unit='us', origin='unix')

In [5]:
events_data = events_data[['event_datetime','event_name', 'platform', 'language', 'user_id','user_pseudo_id', 'tour_id', 'story_id', 'lang_id','audio_time_played','audio_time_paused']]

In [6]:
#  first event per user
first_event = (
    events_data
    .groupby("user_pseudo_id")["event_datetime"]
    .min()
    .reset_index()
    .rename(columns={"event_datetime": "first_event_datetime"})
)

events_data = events_data.merge(first_event, on="user_pseudo_id")

# sort properly
events_data = events_data.sort_values(
    by=["first_event_datetime", "user_pseudo_id", "event_datetime"]
).drop(columns="first_event_datetime")


In [7]:
events_data.head(10)

,event_datetime,event_name,platform,language,user_id,user_pseudo_id,tour_id,story_id,lang_id,audio_time_played,audio_time_paused
1367736,2025-06-30 21:00:33.442000,click_purchases_tab,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,<NA>,<NA>,<NA>,NaN,NaN
650573,2025-06-30 21:00:34.254001,click_purchases_tab,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,<NA>,<NA>,<NA>,NaN,NaN
285708,2025-07-01 06:25:11.609002,SCREEN_VIEW,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,<NA>,<NA>,<NA>,NaN,NaN
115002,2025-07-01 06:28:36.532000,click_purchases_tab,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,<NA>,<NA>,<NA>,NaN,NaN
285693,2025-07-01 06:28:36.532001,click_view_tickets_tab,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,<NA>,<NA>,<NA>,NaN,NaN
553001,2025-07-01 06:28:36.532002,SCREEN_VIEW,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,<NA>,<NA>,<NA>,NaN,NaN
828956,2025-07-01 06:28:39.799003,click_listen_now,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,<NA>,5,NaN,NaN
828957,2025-07-01 06:28:51.839001,start_tour,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,<NA>,5,NaN,NaN
553014,2025-07-01 06:28:51.894002,screen_view,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,<NA>,<NA>,<NA>,NaN,NaN
552998,2025-07-01 06:28:52.212003,story_start,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,50606,5,NaN,NaN


In [8]:
events_data.to_parquet(events_path, engine="pyarrow", compression="snappy")